# 🧠 RAG System - Code de la Route Marocain

**Objectif**: Répondre à des questions sur le Code de la Route marocain en utilisant la RAG (Retrieval-Augmented Generation).

**Pipeline**: CSV → Cleaning → Chunking → Embeddings → FAISS → Retrieval → LLM → Answer

---

## 1. 📦 Imports + Data Load

In [2]:
!pip install pandas numpy sentence-transformers faiss-cpu torch transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import re
import unicodedata
import time
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
import faiss
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import gradio as gr

print("✅ Toutes les librairies chargées")
print(f"   Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

✅ Toutes les librairies chargées
   Device: cpu


In [6]:
# Chargement du CSV
try:
    df = pd.read_csv("code_route.csv")
    print(f"✅ Fichier chargé: {len(df)} articles")
except:
    print("⚠️ Fichier non trouvé, création d'un dataset de démonstration...")
    data = {
        'article_id': [1, 2, 3, 4, 5, 6, 7, 8],
        'article_header': [
            'رخصة السياقة', 'السرعة القصوى', 'حزام الأمان', 'الهاتف أثناء القيادة',
            'السياقة في حالة سكر', 'وقوف السيارات', 'أضواء السيارة', 'تجاوز السيارات'
        ],
        'texte_brut': [
            'لا يجوز قيادة مركبة إلا ب رخصة سياقة سارية المفعول. تسلم الرخصة بعد اجتياز اختبار النظرية والاختبار العملي.',
            'السرعة القصوى: في المدينة 50 كم/س، على الطريق السيار 120 كم/س، على الطريق الوطنية 80 كم/س.',
            'حزام الأمان إجباري على جميع المقاعد. المخالفة تعرض لغرامة 150 درهم.',
            'يمنع استعمال الهاتف المحمول باليد أثناء القيادة. الغرامة تتراوح بين 200 و 500 درهم.',
            'القيادة تحت تأثير الكحول: غرامة 2000-10000 درهم وسحب الرخصة من شهر إلى 6 أشهر.',
            'يمنع الوقوف في أماكن المعاقين وممرات الطوارئ. الغرامة 300 درهم.',
            'يجب تشغيل الأضواء عند حلول الليل وعند تدني الرؤية.',
            'يمنع التجاوز في المنعرجات والمجاز وعند عدم كفاية الرؤية.'
        ],
        'amende': [0, 0, 150, 300, 5000, 300, 100, 400],
        'points': [0, 2, 0, 2, 6, 1, 0, 3]
    }
    df = pd.DataFrame(data)
    print(f"✅ Dataset créé: {len(df)} articles")

df.head(4)

✅ Fichier chargé: 1 articles


,article_id,infraction_desc,categorie_vehicule,amende_fixe,amende_min,amende_max,points_retrait,mots_cles,type_article,categorie_infraction,book,section,chapter,subsection
0,1,لا يجوز لأي شخص أن يسوق مركبة ذات محرك أو مجمو...,"moto, vehicule_agricole, remorque",NaN,NaN,NaN,16,"speed, parking, alcohol, drogue, night, autoro...",definition,cluster_0,NaN,NaN,NaN,NaN


## 2. 🧹 Preprocessing

In [8]:
import re
import pandas as pd

def clean_text(text):
    """Nettoie le texte (arabe + français léger)"""
    if not isinstance(text, str):
        return ""

    # Remove Arabic diacritics
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)

    # Normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# 🎯 Your real text column
TEXT_COL = 'infraction_desc'

# 🚨 Safety check
if TEXT_COL not in df.columns:
    raise ValueError(f"❌ Missing column: {TEXT_COL}. Available: {df.columns.tolist()}")

# 🧼 Clean main text
df['texte_clean'] = df[TEXT_COL].apply(clean_text)

# 🧼 Clean optional fields
df['mots_cles_clean'] = df['mots_cles'].apply(clean_text) if 'mots_cles' in df.columns else ""
df['type_article_clean'] = df['type_article'].apply(clean_text) if 'type_article' in df.columns else ""

# 🧹 Remove empty or too-short rows
df = df[df['texte_clean'].str.len() > 15].reset_index(drop=True)

# 📊 Summary
print(f"✅ Articles after cleaning: {len(df)}")

print("\n📌 Columns used:")
print(f"- Text: {TEXT_COL}")
print(f"- Keywords: mots_cles")
print(f"- Type: type_article")

print("\n🔎 Example cleaned text:")
print(df['texte_clean'].iloc[0][:200] + "...")

✅ Articles after cleaning: 1

📌 Columns used:
- Text: infraction_desc
- Keywords: mots_cles
- Type: type_article

🔎 Example cleaned text:
لا يجوز لأي شخص أن يسوق مركبة ذات محرك أو مجموعة مركبات على الطريق العمومية ما لم يكن حاصلا على رخصة للسياقة سارية الصلاحية ومسلمة من قبل الإدارة. تناسب صنف المركبة أو مجموعة المركبات التي يسوقها. سنة...


## 3. ✂️ Chunking

In [10]:
def chunk_text(text, max_len=400, overlap=80):
    """Découpe un texte en chunks avec overlap"""
    if not isinstance(text, str) or len(text) == 0:
        return []

    if len(text) <= max_len:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + max_len, len(text))
        chunks.append(text[start:end])

        if end == len(text):
            break

        start += max_len - overlap

    return chunks


chunks = []

for _, row in df.iterrows():

    text_chunks = chunk_text(row['texte_clean'])

    for i, chunk in enumerate(text_chunks):
        chunks.append({
            'article_id': row['article_id'],
            'chunk_id': f"{row['article_id']}_{i}",
            'text': chunk,

            # 🧠 SAFE FALLBACKS (no KeyError risk)
            'amende': row.get('amende_fixe', 0),
            'points': row.get('points_retrait', 0),
            'categorie': row.get('categorie_infraction', ''),
            'vehicule': row.get('categorie_vehicule', '')
        })

chunks_df = pd.DataFrame(chunks)

# 🧠 Better embedding text (VERY important for RAG quality)
chunks_df['embed_text'] = chunks_df['text'] + " | " + \
                          chunks_df['categorie'].astype(str) + " | " + \
                          chunks_df['vehicule'].astype(str)

print(f"✅ {len(chunks_df)} chunks créés")
print(f"📊 Moyenne: {len(chunks_df)/len(df):.1f} chunks/article")

chunks_df.head(3)

✅ 599 chunks créés
📊 Moyenne: 599.0 chunks/article


,article_id,chunk_id,text,amende,points,categorie,vehicule,embed_text
0,1,1_0,لا يجوز لأي شخص أن يسوق مركبة ذات محرك أو مجمو...,NaN,16,cluster_0,"moto, vehicule_agricole, remorque",لا يجوز لأي شخص أن يسوق مركبة ذات محرك أو مجمو...
1,1,1_1,مدة أقصاها سنة من تاريخ إقامتهم المؤقتة بالمغ...,NaN,16,cluster_0,"moto, vehicule_agricole, remorque",مدة أقصاها سنة من تاريخ إقامتهم المؤقتة بالمغ...
2,1,1_2,رخصتهم للسياقة تطبيقا للفقرات الموالية. يمكن ل...,NaN,16,cluster_0,"moto, vehicule_agricole, remorque",رخصتهم للسياقة تطبيقا للفقرات الموالية. يمكن ل...


## 4. 🧠 Embeddings

In [11]:
# Modèle multilingue (arabe + français)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Génération des embeddings...")
embeddings = model.encode(chunks_df['embed_text'].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print(f"✅ Embeddings: {embeddings.shape}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Génération des embeddings...


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

✅ Embeddings: (599, 384)


## 5. ⚡ Vector Store (FAISS)

In [12]:
# Normalisation pour similarité cosinus
faiss.normalize_L2(embeddings)

# Création de l'index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"✅ Index FAISS: {index.ntotal} vecteurs")
print(f"   Dimension: {embeddings.shape[1]}")

✅ Index FAISS: 599 vecteurs
   Dimension: 384


## 6. 🔎 Retrieval Function

In [14]:
def retrieve(query, k=4):
    """Recherche les chunks les plus pertinents"""

    # Encode query
    query_vec = model.encode([query])
    query_vec = np.array(query_vec).astype('float32')
    faiss.normalize_L2(query_vec)

    # Search FAISS
    scores, indices = index.search(query_vec, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            row = chunks_df.iloc[idx]

            results.append({
                'score': float(score),
                'article_id': row['article_id'],
                'text': row['text'],
                'amende': row['amende'],
                'points': row['points'],
                'categorie': row['categorie'],
                'vehicule': row['vehicule']
            })

    return results


# 🧪 Test
test = retrieve("رخصة السياقة", k=2)

for r in test:
    print(f"Score: {r['score']:.3f} | Article {r['article_id']}")
    print(f"Text: {r['text'][:120]}...\n")

Score: 0.809 | Article 1
Text: دون الحصول على ترخيص أو عدم احترام الشروط الخاصة المحددة ف الترخيص بالنقل الاستثنائي ؛ ح سس ستحتتتته ‏ سس لب سح 4 - دخول...

Score: 0.789 | Article 1
Text:  آللااف (8.000) درهم. كل شخص صدر في حقه مقرر قضائي حائز لقوة الشيء المقضي به أو قرار إداري بتوقيف رخصة السياقة أو بسحها ...



## 7. 🤖 LLM + Prompt

In [15]:
# Chargement du LLM (petit modèle pour rapidité)
print("Chargement du modèle de langage...")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", trust_remote_code=True)
llm = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.float32,
    trust_remote_code=True
)

generator = pipeline("text-generation", model=llm, tokenizer=tokenizer, max_new_tokens=250)
print("✅ LLM chargé")

Chargement du modèle de langage...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ LLM chargé


In [19]:
def build_prompt(query, docs):
    """Construit le prompt pour le LLM"""

    context = ""

    for i, doc in enumerate(docs, 1):
        context += f"""
Document {i} [Article {doc['article_id']}]:
{doc['text']}

Catégorie: {doc['categorie']}
Véhicule: {doc['vehicule']}
Points: {doc['points']}
Amende: {doc['amende']} MAD

---
"""

    prompt = f"""
Tu es un assistant expert du Code de la Route marocain.

Utilise UNIQUEMENT les documents fournis pour répondre.

Contexte:
{context}

Question: {query}

Réponse claire et précise avec références:
"""

    return prompt

## 8. 🧠 RAG Pipeline (MAIN FUNCTION)

In [20]:
def rag_answer(query):
    """Pipeline RAG complet"""

    # 1. Retrieval
    docs = retrieve(query, k=3)

    if not docs:
        return "Aucun document trouvé.", []

    # 2. OOD check
    if docs[0]['score'] < 0.25:
        return "⚠️ Question hors domaine du Code de la Route marocain.", []

    # 3. Prompt
    prompt = build_prompt(query, docs)

    # 4. Generation
    response = generator(prompt)[0]['generated_text']

    # 5. Extract answer
    if "<|im_start|>assistant" in response:
        answer = response.split("<|im_start|>assistant")[-1].strip()
    else:
        answer = response.replace(prompt, "").strip()

    # 6. Sources (FIXED)
    sources = [
        f"Art.{d['article_id']} (score: {d['score']:.2f})"
        for d in docs
    ]

    return answer, sources

In [21]:
# Test rapide
q = "ما هي شروط الحصول على رخصة السياقة؟"
print(f"Question: {q}\n")
answer, sources = rag_answer(q)
print(f"Réponse:\n{answer}\n")
print(f"Sources: {sources}")

Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: ما هي شروط الحصول على رخصة السياقة؟

Réponse:
الجواب الصحيح هو: "في حال"

هذه النقطة تتعلق مباشرة بالمراقبة والحكم بشأن الحصول على رخصة السياقة. بدلاً من تقديم جدول محدد، يمكن القول أن هذه النقطة تعطي فكرة عن الاحتياجات الأساسية لممارسة الرخصة السياقية بشكل صحيح. 

المراجع الرئيسية:
- Article 8.000 في دستور المغرب
- Article 96 في دستور المغرب
- Clause 0 in the Moroccan Road Traffic Code
- Clause 0 in the Moroccan Road Traffic Code (Clue)
- Article 8.000 in the Moroccan Road Traffic Code

لذا، الجواب الأكثر دقة وإيجابية هو: "في حال" مع الأخبار ذات الصلة.

Sources: ['Art.1 (score: 0.81)', 'Art.1 (score: 0.78)', 'Art.1 (score: 0.77)']


## 9. 🚫 OOD Detection (simple)

*Déjà intégrée dans le pipeline: score < 0.25 = hors domaine*

## 10. 🧪 Testing

In [22]:
print("="*60)
print("TESTS DU SYSTÈME RAG")
print("="*60)

test_questions = [
    ("Arabe 1", "ما هي شروط الحصول على رخصة السياقة؟"),
    ("Arabe 2", "كم تبلغ غرامة استعمال الهاتف أثناء القيادة؟"),
    ("Français 1", "Quelle est la vitesse maximale sur l'autoroute?"),
    ("Français 2", "Quelle est l'amende pour conduite sans ceinture?"),
    ("OOD", "Qui a gagné la Coupe du Monde 2022?"),
]

for name, q in test_questions:
    print(f"\n{'─'*50}")
    print(f"📝 {name}: {q}")
    answer, sources = rag_answer(q)
    print(f"💬 Réponse: {answer[:200]}..." if len(answer) > 200 else f"💬 Réponse: {answer}")
    if sources:
        print(f"📚 Sources: {sources}")

Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TESTS DU SYSTÈME RAG

──────────────────────────────────────────────────
📝 Arabe 1: ما هي شروط الحصول على رخصة السياقة؟


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💬 Réponse: La règles pour obtenir une réciproque d'une licence de route sont détaillées dans les articles 7 et 8 de la Code de la Route marocain. Voici les principales conditions à considérer :

1. **Requêtes de...
📚 Sources: ['Art.1 (score: 0.81)', 'Art.1 (score: 0.78)', 'Art.1 (score: 0.77)']

──────────────────────────────────────────────────
📝 Arabe 2: كم تبلغ غرامة استعمال الهاتف أثناء القيادة؟


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💬 Réponse: الغريزة المطلوبة لإيجاد الرمز هي "catégorie: cluster_0"، حيث يتم البحث عن المعلومات المتعلقة بغرامات الخدمة المصرفية والخدمات الأخرى. في هذا الإطار، يمكننا فهم المعلومات من خلال البحث في المقالات المق...
📚 Sources: ['Art.1 (score: 0.64)', 'Art.1 (score: 0.56)', 'Art.1 (score: 0.55)']

──────────────────────────────────────────────────
📝 Français 1: Quelle est la vitesse maximale sur l'autoroute?


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💬 Réponse: La vitesse maximale sur l'autoroute dépend de plusieurs facteurs tels que le type de véhicule, sa catégorie et son niveau d'engagement dans l'utilisation de l'autoroute. En général, les véhicules agil...
📚 Sources: ['Art.1 (score: 0.72)', 'Art.1 (score: 0.69)', 'Art.1 (score: 0.66)']

──────────────────────────────────────────────────
📝 Français 2: Quelle est l'amende pour conduite sans ceinture?


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💬 Réponse: Le texte mentionne que le permis d'autoriser à conduire sans couteau n'est pas illégal. Dans le cas où le conducteur ne porte pas une couteau lorsqu'il conduit sur les routes, il est considéré comme a...
📚 Sources: ['Art.1 (score: 0.76)', 'Art.1 (score: 0.73)', 'Art.1 (score: 0.71)']

──────────────────────────────────────────────────
📝 OOD: Qui a gagné la Coupe du Monde 2022?
💬 Réponse: Je ne peux pas fournir une réponse à cette question car elle est impossible de trouver dans les données fournies. Les données sont des articles de code de la route marocain, qui n'ont aucune informati...
📚 Sources: ['Art.1 (score: 0.47)', 'Art.1 (score: 0.28)', 'Art.1 (score: 0.26)']


## 11. 🌐 Simple Interface (Gradio)

In [23]:
def gradio_rag(question):
    """Interface Gradio simplifiée"""
    if not question.strip():
        return "Veuillez entrer une question.", ""

    answer, sources = rag_answer(question)
    sources_text = "\n".join(sources) if sources else "Aucune source trouvée"

    return answer, sources_text

# Création de l'interface
with gr.Blocks(title="Code de la Route - Assistant RAG") as demo:
    gr.Markdown("""
    # 🚗 Assistant Code de la Route Marocain

    Posez vos questions en arabe ou en français.
    """)

    question = gr.Textbox(label="Votre question", lines=2, placeholder="Ex: ما هي شروط رخصة السياقة ?")

    with gr.Row():
        answer = gr.Textbox(label="Réponse", lines=10, interactive=False)
        sources = gr.Textbox(label="Sources (articles)", lines=10, interactive=False)

    submit = gr.Button("🔍 Interroger", variant="primary")
    submit.click(gradio_rag, inputs=question, outputs=[answer, sources])

    gr.Examples(
        examples=[
            "ما هي شروط الحصول على رخصة السياقة؟",
            "كم تبلغ غرامة استعمال الهاتف؟",
            "Quelle est la vitesse maximale en ville?",
            "ما هي عقوبة القيادة في حالة سكر؟",
        ],
        inputs=question
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://08b4f9b02a14ea0d91.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## ✅ FIN DU NOTEBOOK

**Le système RAG est fonctionnel!**

Ce qu'il fait:
1. 📂 Charge les données du Code de la Route
2. 🧹 Nettoie les textes arabes
3. ✂️ Découpe les articles en petits chunks
4. 🧠 Génère des embeddings vectoriels
5. ⚡ Stocke tout dans FAISS pour recherche rapide
6. 🔎 Trouve les articles pertinents pour chaque question
7. 🤖 Utilise un LLM pour générer une réponse précise
8. 🚫 Rejette les questions hors domaine (score < 0.25)
9. 🌐 Interface web simple pour interagir

---